# Leçon 12 - Réduction de l'historique du chat avec un bloc-notes d'agent

Ce carnet montre comment gérer le contexte dans les conversations longues en utilisant Microsoft Agent Framework. Au fur et à mesure que les conversations s'allongent, le nombre de jetons augmente — dépassant finalement la fenêtre de contexte du modèle. Nous résolvons ce problème avec un **modèle de synthèse de contexte** et un **bloc-notes d'agent** pour une mémoire persistante.

## Ce que vous apprendrez :
1. **Pourquoi la gestion du contexte est importante** : Comprendre les limites des jetons et les fenêtres de contexte
2. **Agents conscients du contexte** : Construire des agents qui gèrent leur propre contexte de conversation
3. **Modèle de synthèse de contexte** : Utiliser des outils pour condenser l'historique de la conversation
4. **Bloc-notes d'agent** : Mémoire persistante qui survit à la réduction du contexte

## Prérequis :
- Configuration Azure OpenAI avec les variables d'environnement définies
- Compréhension des concepts de base des agents vus dans les leçons précédentes


## Configuration


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Azure AI Foundry client configured")

## Pourquoi la gestion du contexte est importante

Chaque LLM dispose d'une **fenêtre de contexte** finie — le nombre maximal de tokens qu'il peut traiter en une seule requête. Au fur et à mesure qu'une conversation multi-tours progresse :

- Le **nombre de tokens croît linéairement** avec chaque message utilisateur et chaque réponse de l'assistant.
- Les **tokens du prompt dominent le coût** car tout l'historique est renvoyé à chaque tour.
- Finalement, la conversation **dépasse la fenêtre de contexte** et le modèle tronque ou génère une erreur.

### Stratégies de gestion du contexte

| Stratégie | Fonctionnement | Compromis |
|---|---|---|
| **Troncature** | Supprimer les messages les plus anciens | Perte du contexte initial |
| **Résumé** | Condenser les anciens messages en un résumé | Perte de certains détails, mais les points clés sont conservés |
| **Bloc-notes / Mémoire externe** | Stocker les faits clés en dehors de la conversation | Nécessite des appels d’outils, mais survit à toute réduction |

Dans ce notebook, nous combinons le **résumé** avec un **outil bloc-notes** afin que l'agent puisse maintenir la continuité même lorsque l'historique de la conversation est condensé.


## Création d’un agent contextuel


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simuler une longue conversation

Parcourons une conversation à plusieurs échanges pour voir comment le contexte s'accumule. L'agent doit conserver les détails clés (préférences, budget, dates de voyage) au fil des échanges et démontrer la continuité.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Remarquez comment l’agent conserve le contexte des tours précédents — il sait ce qu’est le Japon, le sushi, les temples, la photographie, le budget de 3000 $, le voyage en solo et la période d’avril. Dans une conversation courte, cela fonctionne bien, mais à mesure que la conversation s’allonge, l’historique complet devient coûteux à renvoyer.

Continuons la conversation avec plus de tours pour voir l’accumulation du contexte :


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Modèle de Résumé de Contexte

Au fur et à mesure que la conversation progresse, nous pouvons utiliser un **outil de résumé** pour condenser le contexte accumulé en un format compact. L'agent appelle cet outil pour enregistrer les préférences clés afin que même si les messages plus anciens sont supprimés, l'information essentielle soit préservée.

Ce modèle est la base pour une réduction d'historique plus sophistiquée :
1. L'agent identifie les faits clés de la conversation
2. Il appelle l'outil de résumé pour les conserver
3. Les messages plus anciens peuvent être supprimés en toute sécurité car le résumé capture ce qui est important

Ci-dessous, nous définissons un outil `summarize_preferences` que l'agent peut appeler pour enregistrer un résumé compact de ce qu'il a appris.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Résumé

Dans cette leçon, vous avez appris à gérer le contexte dans les conversations longues avec un agent en utilisant Microsoft Agent Framework :

### Concepts clés
- **Les fenêtres de contexte sont limitées** — chaque jeton dans l’historique de la conversation coûte de l’argent et compte dans la limite.
- **Les outils de synthèse** permettent à l’agent de condenser le contexte accumulé en résumés compacts, réduisant l’utilisation des jetons tout en préservant les informations essentielles.
- **Les carnets de notes de l’agent** fournissent une mémoire externe persistante qui survit à toute réduction de conversation.

### Ce que vous avez construit
- Un **agent conscient du contexte** qui maintient la continuité à travers des conversations à plusieurs tours
- Un **outil de synthèse** (`summarize_preferences`) qui enregistre les détails clés de l’utilisateur dans un format compact
- Une **conversation à plusieurs tours** démontrant la rétention de contexte et la gestion des changements

### Applications dans le monde réel
- **Bots de service client** : se souvenir des préférences au cours de longues sessions de support
- **Assistants personnels** : suivre des projets en cours sans réexpliquer le contexte
- **Tuteurs éducatifs** : maintenir la progression des étudiants à travers de nombreuses interactions

### Étapes suivantes
- Implémenter un outil de carnet de notes complet avec persistance basée sur des fichiers
- Ajouter une troncature automatique de l’historique après la synthèse
- Combiner avec des bases de données vectorielles pour une recherche sémantique en mémoire
- Construire des agents capables de reprendre des conversations des jours plus tard avec tout le contexte


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
